# CatBoost — Ordered Boosting

CatBoost is another **gradient boosting algorithm**, but its main idea is to make boosting work better—especially when the dataset contains **categorical features**.

The first important concept is **Ordered Boosting**.

---

## 1. What problem does CatBoost solve?

In ordinary boosting, each new tree is trained using information from the current model's errors.

Suppose we have:

$$\hat{y}^{(t-1)}$$

Then the next tree learns a correction:

$$\hat{y}^{(t)}=\hat{y}^{(t-1)}+\eta f_t(X)$$

The problem is that when calculating the information used to train the next tree, the model can end up using information that is influenced by the same training example.

This can create a form of **target leakage / prediction shift**.

CatBoost's **Ordered Boosting** is designed to reduce this problem.

---

# 2. Basic idea

CatBoost creates a **random permutation** of the training data.

Example:

Original:

| Row | Feature | Target |
|---|---:|---:|
| 1 | A | 10 |
| 2 | B | 20 |
| 3 | A | 30 |
| 4 | C | 40 |
| 5 | B | 50 |

One random permutation could be:

$$[3,1,5,2,4]$$

Now CatBoost processes observations according to this ordering.

For a particular observation, CatBoost tries to calculate the information needed for that observation using **only preceding observations** in the permutation.

So conceptually:

$$\text{Information for row } i \leftarrow \text{previous rows only}$$

rather than:

$$\text{Information for row } i \leftarrow \text{all training rows including itself}$$

---

# 3. Why is this important?

Consider target encoding.

Suppose:

| City | Target |
|---|---:|
| Mumbai | 1 |
| Mumbai | 0 |
| Mumbai | 1 |

If we calculate Mumbai's target mean for **every row** using all three rows:

$$\frac{1+0+1}{3}=0.667$$

But the first Mumbai row's own target is included in the statistic used to represent that row.

That means the row's target has influenced its own representation.

This is undesirable because the model gets information about the answer while trying to predict the answer.

---

# 4. Ordered approach

Suppose the permutation is:

$$[R_1,R_2,R_3]$$

For $R_1$:

There are no previous observations.

For $R_2$:

Use information from $R_1$ only.

For $R_3$:

Use information from $R_1,R_2$.

So:

| Row | Previous observations used |
|---|---|
| $R_1$ | None |
| $R_2$ | $R_1$ |
| $R_3$ | $R_1,R_2$ |

The current row's target is **not used to calculate its own statistic**.

---

# 5. Target-statistic example

Suppose:

| Row | City | Target |
|---|---|---:|
| R1 | Mumbai | 1 |
| R2 | Mumbai | 0 |
| R3 | Mumbai | 1 |
| R4 | Pune | 0 |

Permutation:

$$R_1 \rightarrow R_2 \rightarrow R_3 \rightarrow R_4$$

For R2:

Previous Mumbai observations:

$$R_1 = 1$$

So its statistic is based on:

$$\frac{1}{1}=1$$

For R3:

Previous Mumbai observations:

$$R_1,R_2$$

Therefore:

$$\frac{1+0}{2}=0.5$$

Notice:

**R3's own target = 1 is not used.**

That's the important mechanism.

---

# 6. But what about the first observation?

There are no previous observations for R1.

CatBoost uses **prior/smoothing mechanisms** so that the statistic does not become undefined or extremely unstable.

Conceptually:

$$\text{Statistic}=\frac{\text{previous target sum}+\text{prior contribution}}{\text{previous count}+\text{prior weight}}$$

The exact implementation is more sophisticated, but the important idea is:

> Early observations don't simply get an unreliable statistic based on zero previous samples.

---

# 7. Ordered Boosting vs ordinary boosting

### Traditional approach

Information for an observation can be influenced by the entire training dataset:

$$\text{row}_i \rightarrow \text{information calculated using all rows}$$

This can create prediction shift.

### CatBoost Ordered Boosting

For each observation:

$$\text{row}_i \rightarrow \text{information from preceding observations}$$

This reduces the dependence of the observation's training information on its own target.

---

# 8. Where does the "Boosting" part come in?

Important: **Ordered Boosting is still boosting.**

CatBoost still builds trees sequentially:

$$F_0(X)$$

then

$$F_1(X)$$

then

$$F_2(X)$$

and so on.

The difference is **how CatBoost obtains the training information for each observation while constructing those boosting iterations**.

So:

```text
Initial model
      ↓
Permutation
      ↓
Ordered statistics / ordered information
      ↓
Build tree
      ↓
Update predictions
      ↓
Next boosting iteration
      ↓
Repeat
```

---

## 9. The key thing to remember

Don't think:

> "CatBoost is a completely different type of boosting."

Think:

> **CatBoost = Gradient Boosting + mechanisms designed to reduce prediction shift, especially through Ordered Boosting and ordered statistics.**

This becomes particularly important when CatBoost handles **categorical features**.

### One-line exam definition

**Ordered Boosting is a CatBoost technique that uses permutation-based, preceding-observation information to reduce target leakage/prediction shift during boosting.**

**Core idea:**

$$\boxed{\text{Current row's target should not directly influence the information used to train that row}}$$

## Difference between XGBoost and CatBoost using an Dataset

### Dataset

Baseline prediction:

$$\hat y_0 = 50$$

So the residual is:

$$r_i=y_i-\hat y_0$$

| Row | Employee | Experience | Actual Salary | Residual |
|---|---|---:|---:|---:|
| 1 | Alice | 1 | 40 | -10 |
| 2 | Bob | 3 | 45 | -5 |
| 3 | Charlie | 5 | 55 | +5 |
| 4 | David | 8 | 60 | +10 |
| 5 | Eva | 10 | 70 | +20 |

The tree wants to learn:

> **Which Experience split gives the best correction to these residuals?**

---

# 1. XGBoost

For this simple regression example, think of the residual as the gradient direction.

So XGBoost sees **all 5 observations when constructing the first tree**:

```text
Alice    → -10
Bob      →  -5
Charlie  →  +5
David    → +10
Eva      → +20
```

It can evaluate possible splits.

For example:

```text
Experience ≤ 4
        /       \
   Alice,Bob   Charlie,David,Eva
```

So:

### Left

$$G_L=-10+(-5)=-15$$

### Right

$$G_R=5+10+20=35$$

Total:

$$G=-15+35=20$$

For squared-error boosting, the leaf correction is related to:

$$w^*=-\frac{G}{H+\lambda}$$

If we simplify with $H=1$ per observation and $\lambda=0$:

Left:

$$w_L=-\frac{-15}{2}=+7.5$$

Right:

$$w_R=-\frac{35}{3}=-11.67$$

So the tree learns:

```text
Experience ≤ 4
       ↓
   correction +7.5

Experience > 4
       ↓
   correction -11.67
```

The exact sign depends on whether we're defining residual as $y-\hat y$ or gradient as $\hat y-y$; the important thing is that the tree uses **all five observations to calculate the split/leaf statistics**.

---

# 2. Now what does CatBoost do differently?

Here's where we need to be precise.

CatBoost creates a **permutation**.

For simplicity:

$$1\rightarrow2\rightarrow3\rightarrow4\rightarrow5$$

So:

```text
Alice → Bob → Charlie → David → Eva
```

Now CatBoost constructs **ordered information**.

For each observation, it doesn't allow that observation to use information derived from its own target.

---

## Alice

Alice is first.

Previous observations:

```text
None
```

So Alice cannot use her own residual:

$$-10$$

to construct her own ordered information.

---

## Bob

Bob has one preceding observation:

```text
Alice → Bob
```

So Bob's ordered information can use:

$$-10$$

but **not Bob's own -5**.

---

## Charlie

Previous:

```text
Alice → Bob → Charlie
```

Charlie can use:

$$-10,-5$$

but **not Charlie's own +5**.

For example, the previous residual average is:

$$\frac{-10+(-5)}{2}=-7.5$$

---

## David

Previous:

```text
Alice → Bob → Charlie → David
```

David can use:

$$-10,-5,+5$$

but not his own $+10$.

Previous average:

$$\frac{-10-5+5}{3}=-3.33$$

---

## Eva

Previous:

```text
Alice → Bob → Charlie → David → Eva
```

Eva can use:

$$-10,-5,+5,+10$$

but not her own $+20$.

Previous average:

$$\frac{-10-5+5+10}{4}=0$$

---

# 3. Put them side by side

This is the important picture.

### XGBoost

When constructing the tree:

```text
All 5 observations
       ↓
┌──────┴──────┐
Alice         Bob
Charlie       David
Eva
       ↓
Calculate split statistics
       ↓
Find best split
```

The training examples participate together in calculating the tree's statistics.

---

### CatBoost Ordered approach

For each observation:

```text
Alice
  ↓
nothing before her

Bob
  ↓
Alice

Charlie
  ↓
Alice + Bob

David
  ↓
Alice + Bob + Charlie

Eva
  ↓
Alice + Bob + Charlie + David
```

So:

$$\boxed{\text{Current observation does not contribute its own target to its ordered information}}$$

---

# 4. But there's an important correction to the mental model

You might now think:

> "So CatBoost doesn't look at all 5 observations when building the tree?"

**No.**

That's not correct.

CatBoost **still trains a tree using the training dataset**.

The difference is **how the training information/predictions used for those observations is generated**.

Think:

### XGBoost

```text
Current model
     ↓
Predictions for all 5
     ↓
Gradients/Hessians for all 5
     ↓
Tree construction
```

### CatBoost Ordered Boosting

```text
Permutation
     ↓
Generate ordered information/predictions
for each observation using preceding data
     ↓
Gradients / training information
     ↓
Tree construction
```

So CatBoost is **not training on only 1, then 2, then 3 observations**.

The whole dataset is still involved in training the tree.

---

# 5. Why does CatBoost do this?

Suppose Eva's target is unusually high:

$$y_{\text{Eva}}=70$$

If Eva's own target can influence the information used to train Eva, the model gets information about the answer while trying to predict that same answer.

CatBoost says:

> For Eva, construct the relevant ordered information without using Eva's target.

So Eva only sees:

```text
Alice
Bob
Charlie
David
   ↓
information for Eva
```

not:

```text
Alice
Bob
Charlie
David
Eva ← own target
```

This reduces **prediction shift**.

---

# 6. The biggest difference

Your original question was:

> **"How does XGBoost look at the 5 observations together?"**

Using this example:

### XGBoost

All five observations contribute to the construction of the current tree's gradient/Hessian statistics:

$$[-10,-5,+5,+10,+20]$$

The tree considers splits such as:

```text
Experience ≤ 4
Experience ≤ 6.5
Experience ≤ 9
```

and evaluates them using the observations on both sides.

### CatBoost

It still uses the complete training set, **but the information associated with an individual observation is generated in an ordered fashion**.

For Eva:

$$[-10,-5,+5,+10]\rightarrow\text{information for Eva}$$

rather than allowing:

$$[-10,-5,+5,+10,+20]\rightarrow\text{information for Eva}$$

where Eva's own target participates.

---

## One sentence to remember

> **XGBoost builds the tree from the training observations' current gradients/Hessians, while CatBoost's Ordered Boosting carefully generates training information for each observation using a permutation and preceding observations, reducing self-influence/prediction shift.**


Yes — you're understanding **Ordered Target Statistics** correctly. The confusing part is that **Ordered Boosting is a different thing**.

Let's separate them completely.

## 1. Ordered Target Statistics

This is for handling a **categorical feature**.

Example:

| Row | City | Salary |
|---|---|---:|
| 1 | Mumbai | 40 |
| 2 | Mumbai | 45 |
| 3 | Pune | 60 |
| 4 | Mumbai | 50 |

For Row 4, City = Mumbai.

Previous Mumbai rows are:

- Row 1 → 40
- Row 2 → 45

So conceptually:

$$\text{Mumbai statistic}=\frac{40+45}{2}=42.5$$

**Yes. This is exactly the idea you understood.**

The important part is:

> **Use previous observations of the same category to create a numerical statistic for the categorical feature.**

---

# 2. Ordered Boosting is different

Ordered Boosting is **not**:

> "Take the average of previous same-category targets."

Instead, it is about **how CatBoost gets the training prediction/error information for each observation during boosting.**

Let's forget categories for a moment.

Suppose:

| Row | Experience | Salary |
|---|---:|---:|
| 1 | 1 | 40 |
| 2 | 3 | 45 |
| 3 | 5 | 55 |
| 4 | 8 | 60 |
| 5 | 10 | 70 |

Permutation:

$$R_1\rightarrow R_2\rightarrow R_3\rightarrow R_4\rightarrow R_5$$

For Row 3, CatBoost's ordered approach makes its training information based on the **preceding observations**, rather than allowing Row 3's own target to influence that information.

So conceptually:

```text
R1 → R2 → R3

R3's information
      ↑
   R1 + R2
```

not:

```text
R1 + R2 + R3
             ↑
        R3's own target
```

The purpose is to reduce **prediction shift / self-influence** during boosting.

---

# The difference in one table

| | Ordered Target Statistics | Ordered Boosting |
|---|---|---|
| **What is it for?** | Categorical features | Boosting/training process |
| **What does it use?** | Previous same-category targets | Previous observations to generate training information |
| **Example** | Mumbai → previous Mumbai salaries | Row 3 → information based on rows before Row 3 |
| **Main purpose** | Convert category into useful numerical information without leakage | Reduce prediction shift |

### Remember this:

**Ordered Target Statistics:**

> "For this category, what happened in previous examples of this category?"

**Ordered Boosting:**

> "For this training observation, don't let its own target influence the information used to train it."

That's the key distinction.

# CatBoost — Target Statistics / Target Encoding

You already understood the basic idea:

> **For a category, use information from previous occurrences of that category to create a numerical value.**

Now let's make the calculation precise.

### Example

| Row | City | Salary |
|---|---|---:|
| R1 | Mumbai | 40 |
| R2 | Mumbai | 45 |
| R3 | Pune | 60 |
| R4 | Mumbai | 50 |
| R5 | Pune | 55 |

Permutation:

$$R_1 \rightarrow R_2 \rightarrow R_3 \rightarrow R_4 \rightarrow R_5$$

For **R4 (Mumbai)**, previous Mumbai salaries are:

$$40,\ 45$$

A simple average would be:

$$\frac{40+45}{2}=42.5$$

But CatBoost doesn't simply use this raw average. It uses a **smoothed target statistic** so that categories with very few previous observations don't produce unstable values.

A simplified form is:

$$\text{CTR}_i=\frac{\sum_{j<i,\ c_j=c_i}y_j+\alpha P}{N_i+\alpha}$$

where:

- $\sum y_j$ = target sum of previous observations with the same category
- $N_i$ = number of previous observations with the same category
- $P$ = prior value
- $\alpha$ = smoothing strength

So for R4:

$$\text{CTR}_{R4}=\frac{40+45+\alpha P}{2+\alpha}$$

The resulting number becomes a **feature value** that the tree can use.

```text
City
 ↓
Ordered Target Statistic
 ↓
Numerical value
 ↓
Tree split
 ↓
Prediction
```

### Why smoothing?

Suppose a category appears only once.

Without smoothing:

$$\text{CTR}=80$$

That single observation could make the category look extremely predictive.

With a prior, CatBoost pulls the value toward a reasonable baseline when there isn't enough historical information.

---

## Classification works similarly

Suppose:

| Row | City | Bought |
|---|---|---:|
| R1 | Mumbai | 1 |
| R2 | Mumbai | 0 |
| R3 | Mumbai | 1 |

For R3, previous Mumbai targets are:

$$1,0$$

Simple target mean:

$$\frac{1+0}{2}=0.5$$

So Mumbai gets an ordered statistic around **0.5** for R3.

The important rule remains:

> **The current row's target is not used to calculate its own ordered statistic.**

This is the core of CatBoost's target encoding.

# CatBoost — Missing Values

CatBoost can handle **missing values (`NaN`) natively**, so you generally don't have to manually replace every missing value before training.

### Example

Suppose:

| Experience | Salary |
|---:|---:|
| 1 | 40 |
| 3 | 45 |
| **NaN** | 50 |
| 8 | 60 |
| 10 | 70 |

Consider a candidate split:

$$\text{Experience} < 5$$

The normal values can be divided:

```text
Experience < 5
→ 1, 3

Experience ≥ 5
→ 8, 10
```

But what happens to the `NaN` row?

CatBoost can **learn how to handle the missing value during tree construction** rather than simply assuming that `NaN = 0` or replacing it with the mean.

Conceptually:

```text
             Experience < 5?
                 /      \
               Yes       No
              /           \
           1, 3          8, 10
                          + NaN
```

The algorithm evaluates how missing values should be treated while considering splits.

### Why is this useful?

Suppose missingness itself contains information.

For example:

```text
Experience = NaN
```

might systematically occur for a particular group of observations.

Replacing every `NaN` with the mean could destroy that information.

CatBoost can instead let the tree learn a useful direction for missing values.

---

## Important distinction from what we just learned

**Target Statistics:**

```text
Categorical value
      ↓
Previous same-category targets
      ↓
Numerical statistic
      ↓
Tree
```

**Missing Values:**

```text
Feature = NaN
      ↓
Tree learns how to handle it
      ↓
Useful branch
```

So CatBoost's main strengths are becoming:

$$\boxed{\text{Categorical Features + Ordered Statistics + Ordered Boosting + Native Missing Values}}$$

# CatBoost — Hyperparameters

These are the main CatBoost hyperparameters you should know. Since you already understand the core mechanism, focus on **what each controls and its effect**.

| Hyperparameter | What it controls |
|---|---|
| `iterations` | Number of boosting trees |
| `learning_rate` | Contribution of each tree |
| `depth` | Maximum depth of each tree |
| `l2_leaf_reg` | L2 regularization on leaf values |
| `loss_function` | Objective/loss being optimized |
| `random_strength` | Randomness added when choosing splits |
| `bagging_temperature` | Controls Bayesian bagging/randomness |
| `subsample` | Fraction of samples used for training when applicable |
| `border_count` | Number of bins/borders for numerical features |
| `cat_features` | Specifies which features are categorical |

### 1. `iterations`

Number of boosting trees.

If:

```python
iterations = 500
```

CatBoost can build up to **500 sequential trees**.

More iterations → potentially better fit, but too many can cause overfitting.

---

### 2. `learning_rate`

Controls how much each new tree changes the prediction:

$$\hat y^{(t)}=\hat y^{(t-1)}+\eta f_t(X)$$

where $\eta$ is the learning rate.

Small learning rate → smaller corrections → usually need more trees.

Large learning rate → larger corrections → fewer trees may be needed, but overfitting can happen faster.

---

### 3. `depth`

Controls tree depth.

```text
depth = 4
```

means relatively shallow trees.

```text
depth = 10
```

allows much more complex trees.

Higher depth → more expressive model → greater overfitting risk.

---

### 4. `l2_leaf_reg`

L2 regularization applied to leaf values.

Conceptually, it penalizes excessively large leaf corrections:

$$\Omega=\lambda\sum_j w_j^2$$

Higher `l2_leaf_reg` → stronger regularization → less aggressive leaf values.

---

### 5. `loss_function`

Specifies what the model is optimizing.

Examples:

```python
loss_function="RMSE"
```

for regression.

```python
loss_function="Logloss"
```

for binary classification.

```python
loss_function="MultiClass"
```

for multiclass classification.

---

### 6. `random_strength`

Controls randomness used during split selection.

Higher value → more randomness in choosing splits.

This can help reduce overfitting.

---

### 7. `bagging_temperature`

Controls the amount of randomness in CatBoost's Bayesian bootstrap/bagging procedure.

Higher value → generally more random sampling weights.

Lower value → more deterministic behavior.

---

### 8. `border_count`

CatBoost also discretizes numerical features into bins/borders.

For example:

```text
Age
 ↓
0–20
21–40
41–60
61–80
```

`border_count` controls how many candidate borders/bins can be used.

Higher → finer representation.

Lower → faster/coarser representation.

---

### What you should remember for practical use

The **most important** ones are:

$$\boxed{\text{iterations,\ learning\_rate,\ depth,\ l2\_leaf\_reg}}$$

And because CatBoost's special strength is categorical data, also understand:

$$\boxed{\text{cat\_features}}$$

That's enough for your current tree-learning roadmap; you don't need to memorize every CatBoost parameter.